# 01 · ReAct Fundamentals (Single-Step)

## 🎯 What You'll Learn
- The core ReAct pattern: Reasoning → Action → Observation
- How to build deterministic tool selection logic
- Reasoning templates and prompt patterns
- Unit testing for agent behavior
- Pretty printing for debugging

## 📋 Shared Legal Dataset
We'll use these contract snippets throughout all notebooks:

```
CONTRACT_A = "EMPLOYMENT AGREEMENT between TechMahindra Solutions and John Doe. Termination: Either party may terminate with 30 days notice. Liability: Company liable for damages up to $50,000."

CONTRACT_B = "SERVICE AGREEMENT between ABC Corp and XYZ Ltd. Payment terms: Net 30 days. Governing law: This agreement shall be governed by Maharashtra state laws."
```

Welcome! Master the single-step ReAct pattern with tiny code snippets and hands-on practice.


## Theory: What is ReAct?
- ReAct lets an agent explicitly write down its thought (reasoning), pick a tool (action), observe a result, and answer.
- We start with a single-step version: one reasoning, one action, one observation.


In [4]:
# Shared dataset (consistent across all notebooks)
CONTRACT_A = "EMPLOYMENT AGREEMENT between TechMahindra Solutions and John Doe. Termination: Either party may terminate with 30 days notice. Liability: Company liable for damages up to $50,000."
CONTRACT_B = "SERVICE AGREEMENT between ABC Corp and XYZ Ltd. Payment terms: Net 30 days. Governing law: This agreement shall be governed by Maharashtra state laws."

# Mini tools (tiny, readable)
from typing import Dict

def tool_search_stub(query: str) -> str:
    # Simple keyword matching against our contracts
    if "termination" in query.lower():
        return f"[search] Found: Either party may terminate with 30 days notice"
    if "payment" in query.lower():
        return f"[search] Found: Payment terms: Net 30 days"
    return f"[search] Searched for: {query}"

def tool_explain_stub(text: str) -> str:
    return f"[explain] In simple words: {text[:60]}..."

def tool_greet_stub(name: str = "Friend") -> str:
    return f"[greet] Hello {name}! Upload a contract and I'll help you analyze it."


## Pattern: Reason → Action → Observation
We encode a tiny single-step loop. The agent decides one tool and returns its observation as the answer.


In [9]:
# Reasoning templates for better consistency
REASONING_TEMPLATES = {
    "greet": "User is greeting me, so I should respond warmly and offer help.",
    "search": "User wants information from the contract, so I need to search for relevant terms.",
    "explain": "User needs clarification, so I should provide a simple explanation."
}

def choose_tool(user_query: str):
    # converting the user query into lowercase
    q = user_query.lower()
    
    # if hi/hello/hey are found in user query, then return greet
    if any(k in q for k in ["hi", "hello", "hey"]):
        return "greet", lambda: tool_greet_stub("Friend")
    
    # if explain/meaning/mean/simplify are found in user query, return explain
    if any(k in q for k in ["explain", "meaning", "mean", "simplify"]):
        return "explain", lambda: tool_explain_stub(user_query)
    
    # else for all the other cases return search
    return "search", lambda: tool_search_stub(user_query)

def react_single_step(user_query: str) -> Dict[str, str]:
    # REASONING: what should I do?
    action, run = choose_tool(user_query)
    reasoning = REASONING_TEMPLATES.get(action, f"I will use {action} to help with: {user_query}")
    
    # ACTION: pick and run a tool
    observation = run()
    
    # RESPONSE: final answer
    answer = observation  # single-step: observation is the answer
    
    return {
        "reasoning": reasoning,
        "action": action,
        "observation": observation,
        "answer": answer
    }

# Pretty printer for debugging
def pretty_print_react(result: Dict[str, str]) -> None:
    print("🧠 REASONING:", result["reasoning"])
    print("⚡ ACTION:", result["action"])
    print("👀 OBSERVATION:", result["observation"])
    print("💬 FINAL ANSWER:", result["answer"])
    print("-" * 50)

# Test it out
result = react_single_step("Hello how are you")
#print(result)
pretty_print_react(result)


🧠 REASONING: User is greeting me, so I should respond warmly and offer help.
⚡ ACTION: greet
👀 OBSERVATION: [greet] Hello Friend! Upload a contract and I'll help you analyze it.
💬 FINAL ANSWER: [greet] Hello Friend! Upload a contract and I'll help you analyze it.
--------------------------------------------------


## Unit Testing Your Agent

Let's test our agent systematically:


In [12]:
# Unit tests for agent behavior
def test_agent():
    test_cases = [
        ("Hello there", "greet"),
        ("What does termination mean?", "explain"), 
        ("Find termination clause", "search"),
        ("Hi", "greet"),
        ("Explain liability", "explain"),
        ("Search payment terms", "search"),
        ("Hey there", "greet"),
        ("What does indemnity mean?", "explain"),
        ("Find the penalty", "search")            
                        
    ]
    
    print("🧪 UNIT TESTS:")
    for query, expected_action in test_cases:
        result = react_single_step(query)
        actual_action = result["action"]
        status = "✅ PASS" if actual_action == expected_action else "❌ FAIL"
        print(f"{status} '{query}' → {actual_action} (expected: {expected_action})")

test_agent()


🧪 UNIT TESTS:
✅ PASS 'Hello there' → greet (expected: greet)
✅ PASS 'What does termination mean?' → explain (expected: explain)
✅ PASS 'Find termination clause' → search (expected: search)
✅ PASS 'Hi' → greet (expected: greet)
✅ PASS 'Explain liability' → explain (expected: explain)
✅ PASS 'Search payment terms' → search (expected: search)
✅ PASS 'Hey there' → greet (expected: greet)
✅ PASS 'What does indemnity mean?' → explain (expected: explain)
✅ PASS 'Find the penalty' → search (expected: search)
